In [2]:
!pip install CosinorPy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from CosinorPy import file_parser, cosinor, cosinor1, cosinor_nonlin


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 322.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 155.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 397.9 MB/s eta 0:00:00
  Created wheel for CosinorPy: filename=CosinorPy-3.1-py3-none-any.whl size=62318 sha256=16efac9161970a8f60c1f75c3acfea59f6c3d07d7de7c7b4e4b702b03ec89935
  Stored in directory: /tmp/pip-ephem-wheel-cache-uzy9pip5/wheels/01/ca/d9/5d0ae79d8b7f7366ea92b2367d77c9fdbdbf7e15ce3352fe2a
Successfully built CosinorPy
ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/app-root/bin/pyaml'
Check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


ModuleNotFoundError: No module named 'CosinorPy'

In [ ]:
highResData = pd.read_csv('./data/high-resolution/md_data_controls.csv')
highResData

In [ ]:
highResData.columns

In [ ]:
highResData['MasterID'].unique()

In [ ]:
def get_time(datetime):
    return datetime.strftime('%H:%M')
    
highResDataUpdated = pd.read_csv('./data/high-resolution/md_data_controls.csv')
highResDataUpdated['datetime'] = pd.to_datetime(highResDataUpdated['SampleTime'], format='%d/%m/%Y %H:%M')
highResDataUpdated['Cortisol'] = highResDataUpdated['Cortisol'].interpolate(method='linear')
highResDataUpdated['18OHF'] = highResDataUpdated['18OHF'].interpolate(method='linear')
highResDataUpdated['Cortisone'] = highResDataUpdated['Cortisone'].interpolate(method='linear')
highResDataUpdated.reset_index(inplace=True)
highResDataUpdated['CortisolGradient'] = np.gradient(highResDataUpdated['Cortisol'])
highResDataUpdated.loc[highResDataUpdated.groupby('MasterID').head(1).index, 'CortisolGradient'] = 0
highResDataUpdated['time'] = highResDataUpdated['datetime'].dt.time
highResDataUpdated['timeFloat'] = highResDataUpdated['datetime'].dt.hour + highResDataUpdated['datetime'].dt.minute / 60.0

highResDataUpdated = highResDataUpdated.sort_values(by='datetime')
highResDataUpdated[highResDataUpdated['MasterID'] == 71]

In [ ]:
highResDataUpdated['datetime'].unique()

In [ ]:
patient_ids = highResDataUpdated['MasterID'].unique()
patient_ids


In [ ]:
x_lims = highResDataUpdated['datetime'].unique()
x_lim = (x_lims[0], x_lims[-1])

figsize = (16,3)
figsize2 = (16,4)



def plot_group(group, userid, figsize, figsize2, x_lim):
    fig, ax = plt.subplots(figsize=figsize)
    group['Cortisol'].idxmax()
    x = group['datetime']
    y1 = group['Cortisol']
    y2 = np.gradient(group['Cortisol'])
    # Rolling average
    y3 = group['Cortisol'].transform(lambda x: x.rolling(10, 1).mean())
    
    max_time = group.loc[group['Cortisol'].idxmax()]['datetime']
    min_time = group.loc[group['Cortisol'].idxmin()]['datetime']
    
    # Set x-axis limits / labels
    ax.set_xlim(x_lims[0], x_lims[-1])
    ax.set_xlabel('Time')
    ax.set_xticks(x_lims[::5])  # Adjust the step as needed for better readability
    ax.set_xticklabels(x_lims[::5], rotation=45, ha='right')
    ax.set_ylabel('Cortisol Levels')
    
    # Plotting the data
    ax.plot(x, y1, color='blue')
    ax.bar(x, y2, color='orange', label='Cortisol Gradient', width=0.01)
    ax.plot(x, y3, color='green', label='Cortisol Rolling Average', linewidth=2)
    fig.suptitle(f'Cortisol Levels and Gradient Over Time for User {userid}. Max: {max_time}. Min: {min_time}')
    ax.grid(True)

for _userid, _group in highResDataUpdated.groupby('MasterID'):
    group = _group.copy()
    userid = _userid
    plot_group(group, userid, figsize, figsize2, x_lim)

# plot_group(group, userid, figsize, figsize2, x_lim)
plt.show()
# plt.plot(group['datetime'], group['Cortisol'], marker='o', label=f'User {userid}')



In [ ]:
for _userid, _group in highResDataUpdated.groupby('MasterID'):
    print(_userid, _group['Cortisol'].count())

In [ ]:
hour, min = highResDataUpdated['time']
hour

In [ ]:
highResDataUpdated['timeFloat'] = highResDataUpdated['datetime'].dt.hour + highResDataUpdated['datetime'].dt.minute / 60.0

cortisol_df = highResDataUpdated[['timeFloat','time','Cortisol']].groupby('time').mean().reset_index()
print(cortisol_df)
cortisol_df['rolling_average'] = cortisol_df['Cortisol'].rolling(window=10, min_periods=1).mean()
# ax= cortisol_df.plot(x='time', y='Cortisol', title='Mean Cortisol Levels Over Time')

cortisol_df2 = highResDataUpdated[['time','Cortisol']].groupby('time').median().reset_index()
# cortisol_df2.plot(x='time', y='Cortisol', title='Median Cortisol Levels Over Time')

print(f'Min mean cortisol: {cortisol_df["Cortisol"].min()}.  Max mean cortisol: {cortisol_df["Cortisol"].max()}')


def get_parameters(cortisol_df):
    '''
    Calculate the parameters of the cortisol rhythm from the dataframe.
    :param cortisol_df: 
    :return: {'mesor': float, 'amplitude', float, 'peak': float, 'nadir': nadir, 'peak_idx', int}
    '''
    nadir = cortisol_df['Cortisol'].min() # lowest point of the curve
    peak = cortisol_df['Cortisol'].max() # highest point of the curve
    mesor = (nadir + peak) / 2 # midpoint of the curve
    amplitude = peak - mesor # amplitude of the curve (from midpoint to highest/lowest point)
    peak_idx = cortisol_df['Cortisol'].idxmax() # index of the peak cortisol value
    nadir_idx = cortisol_df['Cortisol'].idxmin() # index of the nadir cortisol value
    
    return {
        'mesor': mesor,
        'amplitude': amplitude,
        'nadir': nadir, 
        'peak': peak,
        'peak_idx': peak_idx,
        'nadir_idx': nadir_idx
    }

def get_initial_parameters(cortisol_df):
    '''
    Get the initial parameters for the cosinor function.
    :param cortisol_df: 
    :return: 
    '''
    params = get_parameters(cortisol_df)
    
    
    return [
        5.6,
        5.6,
        9.4,
        2.5,
        3.0,
        2.0,
        1.0,
        1.1
    ]

initial_params = get_initial_parameters(highResDataUpdated)  # Initial guess for MESOR, Amplitude, Acrophase

def cosinor_function(t, M, A1, A2, A3, phi1, phi2, phi3, e):
    '''
    Cosinor function to model the cortisol rhythm.

    :param M: MESOR (Midline Statistic Of Rhythm, a rhythm-adjusted mean) 
    :param A: Amplitude - mesaure of half the extent of predictable variation in the cycle
    :param t: time index (from 0 to tau)
    :param phi: acrophase in radians (a measure of the time of overall high values in each cycle)
    :param e: error term
    :return: float
    '''
    tau_1 = 24
    tau_2 = 12
    tau_3 = 8

    return (M 
            + A1 * np.cos(((2 * np.pi * t) / tau_1) + phi1)
            + A2 * np.cos(((2 * np.pi * t) / tau_2) + phi2)
            + A3 * np.cos(((2 * np.pi * t) / tau_3) + phi3)
            + e * t)

# def cosinor_function(t, M, A1, A2, phi1, phi2, e):
#     '''
#     Cosinor function to model the cortisol rhythm.
#     
#     :param M: MESOR (Midline Statistic Of Rhythm, a rhythm-adjusted mean) 
#     :param A: Amplitude - mesaure of half the extent of predictable variation in the cycle
#     :param t: time index (from 0 to tau)
#     :param phi: acrophase in radians (a measure of the time of overall high values in each cycle)
#     :param e: error term
#     :return: float
#     '''
#     tau_1 = 24
#     tau_2 = 8
#     
#     return (M 
#             + A1 * np.cos(((2 * np.pi * t) / tau_1) + phi1)
#             + A2 * np.cos(((2 * np.pi * t) / tau_2) + phi2)
#             + e * t)
            

def get_cosine_values(cortisol_df, initial_params = initial_params):
    df = cortisol_df.sort_values(by='timeFloat')
    x = df['timeFloat'].values  # time in mins
    y = df['Cortisol'].values  # cortisol values
    
    
    params, params_covariance = curve_fit(cosinor_function, x, y, p0=initial_params)
    
    plt.scatter(x, y, label='Data Points', color='blue',s=0.1)
    plt.plot(x, cosinor_function(x, *params))
    plt.show()
    
    return params, params_covariance

params, params_covariance = get_cosine_values(highResDataUpdated)

original_params = get_parameters(cortisol_df)
mesor = original_params['mesor']
amplitude = original_params['amplitude']
print(f'Original Parameters: {original_params}')
print(f'Initial Parameters: {initial_params}')
print(f'Original Mesor: {mesor}, Amplitude: {amplitude}')
print(f'MESOR: {params[0]}, Amplitude: {params[1]}, Acrophase: {params[2]}')

# highResDataUpdated['Cortisol'].values

In [ ]:
highResDataUpdated

In [ ]:
test_pd = highResDataUpdated.copy()
test_pd.sort_values(by='timeFloat', inplace=True)

data = {
    'test': test_pd['MasterID'].values,
    'x': test_pd['timeFloat'].values,
    'y': test_pd['Cortisol'].values
}

# We want the order the columns, so lets specify in columns parameter
df = pd.DataFrame(data, columns=['test','x','y'])
df_results = cosinor_nonlin.fit_generalized_cosinor_group(df, period = 24, plot=True)#, folder="nonlin_gen1_models") 
